# GRAPES

**Paper:** Younesian et al., *GRAPES: Learning to Sample Graphs for Scalable Graph Neural Networks*, TMLR 2024, [arXiv:2310.03399](https://arxiv.org/abs/2310.03399)

## Kiểm tra GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    raise RuntimeError('Chưa bật GPU!')

## Clone repo

In [ ]:
!git clone https://github.com/dfdazac/grapes.git
%cd grapes
!ls

## Cài dependencies

In [ ]:
import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '')[:3] if torch.cuda.is_available() else 'cpu'
PYG_URL = f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
print(f'torch={TORCH}, cuda={CUDA}')

!pip install typed-argument-parser==1.8.0 ogb==1.3.6 wandb psutil tqdm -q
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse -f {PYG_URL} -q

import torch_geometric, ogb, torch_sparse, tap, psutil, wandb
print('torch_geometric:', torch_geometric.__version__)
print('ogb:', ogb.__version__)
print('Dependencies OK')

## Patch compatibility

- **Patch 1–2** (`modules/utils.py`): scipy >= 1.14 không nhận `torch.Tensor` làm index sparse matrix → thêm `.numpy()`
- **Patch 3** (`ogb/nodeproppred/dataset_pyg.py`): PyTorch >= 2.6 đổi default `weights_only=True` → thêm `weights_only=False`

In [ ]:
import re, glob

# ── Patch 1 & 2: modules/utils.py ──
utils_path = '/content/grapes/modules/utils.py'
with open(utils_path, 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines):
    if 'adjacency[nodes].tocoo()' in line:
        lines[i] = line.replace('adjacency[nodes]', 'adjacency[nodes.numpy()]')
        print(f'Patched utils.py line {i+1}: get_neighborhoods')
    if 'row_slice = adjacency[rows]' in line:
        lines[i] = line.replace('adjacency[rows]', 'adjacency[rows.numpy()]')
        print(f'Patched utils.py line {i+1}: slice_adjacency rows')
    if 'row_col_slice = row_slice[:, cols]' in line:
        lines[i] = line.replace('row_slice[:, cols]', 'row_slice[:, cols.numpy()]')
        print(f'Patched utils.py line {i+1}: slice_adjacency cols')
with open(utils_path, 'w') as f:
    f.writelines(lines)

# ── Patch 3 & 5: ogb/nodeproppred/dataset_pyg.py ──
ogb_files = glob.glob('/usr/local/lib/python3.*/dist-packages/ogb/nodeproppred/dataset_pyg.py')
assert ogb_files, 'Không tìm thấy dataset_pyg.py'
ogb_path = ogb_files[0]
with open(ogb_path, 'r') as f:
    ogb_lines = f.readlines()

for i, line in enumerate(ogb_lines):
    # Patch 3: torch.load → weights_only=False
    if 'torch.load(' in line and 'weights_only' not in line:
        ogb_lines[i] = re.sub(
            r'torch\.load\(([^)]+)\)',
            lambda m: f'torch.load({m.group(1)}, weights_only=False)',
            line
        )
        print(f'Patched dataset_pyg.py line {i+1}: weights_only=False')
    # Patch 5: thay cả dòng if input(...) == 'y': → if False:
    if 'input(' in line and 'update' in line and "== 'y'" in line:
        indent = len(line) - len(line.lstrip())
        ogb_lines[i] = ' ' * indent + 'if False:  # patched: skip update prompt\n'
        print(f'Patched dataset_pyg.py line {i+1}: skip update prompt')

with open(ogb_path, 'w') as f:
    f.writelines(ogb_lines)

# ── Patch 4: ogb/utils/url.py — decide_download luôn True ──
url_files = glob.glob('/usr/local/lib/python3.*/dist-packages/ogb/utils/url.py')
assert url_files, 'Không tìm thấy ogb/utils/url.py'
url_path = url_files[0]
with open(url_path, 'r') as f:
    url_lines = f.readlines()
for i, line in enumerate(url_lines):
    if 'def decide_download' in line and 'patched' not in line:
        url_lines.insert(i + 1, '    return True  # patched: always download\n')
        print(f'Patched url.py line {i+1}: decide_download always True')
        break
with open(url_path, 'w') as f:
    f.writelines(url_lines)

# ── Verify syntax ──
import py_compile
for path in [utils_path, ogb_path, url_path]:
    try:
        py_compile.compile(path, doraise=True)
        print(f'  OK syntax: {path.split("/")[-1]}')
    except py_compile.PyCompileError as e:
        print(f'  SYNTAX ERROR: {e}')

print('\nAll patches done.')

## Hàm chạy thực nghiệm

In [ ]:
import subprocess, re, json, time, datetime
import numpy as np

RESULTS = {}

def run_grapes(config_file, label, runs=3):
    print(f'\n{"="*60}')
    print(f'  {label}  |  {config_file}  |  runs={runs}')
    print(f'{"="*60}')
    cmd = [
        'python', 'main.py',
        f'--config_file={config_file}',
        f'--runs={runs}',
        '--log_wandb=False',
    ]
    print('CMD:', ' '.join(cmd))
    t0 = time.time()
    out = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd='/content/grapes',
        input='n\ny\nn\ny\n',
    )
    elapsed = time.time() - t0

    if out.stdout:
        print(out.stdout[-4000:] if len(out.stdout) > 4000 else out.stdout)
    if out.returncode != 0:
        print('--- STDERR ---')
        print(out.stderr[-2000:])
        return None

    # Parse "Acc: XX.XX ± XX.XX"
    for line in reversed(out.stdout.splitlines()):
        m = re.search(r'Acc:\s*([\d.]+)\s*[±\+\-]+\s*([\d.]+)', line)
        if m:
            mean = float(m.group(1)) / 100
            std  = float(m.group(2)) / 100
            print(f'\n  ✓ {label}: {mean:.4f} ± {std:.4f}  ({elapsed:.0f}s)')
            return {'mean': mean, 'std': std, 'time_s': round(elapsed, 1)}

    # Fallback: dòng test_f1
    for line in reversed(out.stdout.splitlines()):
        if any(k in line.lower() for k in ['test_f1', 'test_accuracy']):
            nums = re.findall(r'\d+\.\d+', line)
            if nums:
                mean = float(nums[-1])
                print(f'\n  ✓ {label} (fallback): {mean:.4f}  ({elapsed:.0f}s)')
                return {'mean': mean, 'std': 0.0, 'time_s': round(elapsed, 1)}

    print('  ✗ Không parse được kết quả.')
    return None

## Cora

In [ ]:
r = run_grapes('configs/gflownet/cora.txt', 'GRAPES / Cora', runs=3)
if r: RESULTS['cora_grapes'] = r

r = run_grapes('configs/random/cora.txt', 'Random / Cora', runs=3)
if r: RESULTS['cora_random'] = r

print('\n=== Kết quả hiện tại ===')
print(json.dumps(RESULTS, indent=2))

## CiteSeer

In [ ]:
r = run_grapes('configs/gflownet/citeseer.txt', 'GRAPES / CiteSeer', runs=3)
if r: RESULTS['citeseer_grapes'] = r

r = run_grapes('configs/random/citeseer.txt', 'Random / CiteSeer', runs=3)
if r: RESULTS['citeseer_random'] = r

print('\n=== Kết quả hiện tại ===')
print(json.dumps(RESULTS, indent=2))

## ogbn-products (bỏ qua nếu OOM)

In [ ]:
r = run_grapes('configs/gflownet/ogbn-products.txt', 'GRAPES / ogbn-products', runs=1)
if r: RESULTS['ogbn_products_grapes'] = r

## ogbn-arxiv

In [ ]:
r = run_grapes('configs/gflownet/ogbn-arxiv.txt', 'GRAPES / ogbn-arxiv', runs=3)
if r: RESULTS['ogbn_arxiv_grapes'] = r

r = run_grapes('configs/random/ogbn-arxiv.txt', 'Random / ogbn-arxiv', runs=3)
if r: RESULTS['ogbn_arxiv_random'] = r

print('\n=== Kết quả hiện tại ===')
print(json.dumps(RESULTS, indent=2))

## Tổng hợp

In [ ]:
import json, datetime, torch

print(f'\n{"="*60}')
print(f'{"Dataset":<22} {"Method":<10} {"Acc":>10} {"± Std":>8}')
print(f'{"-"*60}')
for key, val in RESULTS.items():
    ds, method = key.rsplit('_', 1)
    print(f'{ds:<22} {method:<10} {val["mean"]:>10.4f} {val["std"]:>8.4f}')
print(f'{"="*60}')

output = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'torch_version': torch.__version__,
    'results': RESULTS
}
with open('/content/grapes/grapes_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nSaved: grapes_results.json')

## Tải file về máy

In [ ]:
from google.colab import files
files.download('/content/grapes/grapes_results.json')